# Start here

This workbench holds the same five steps as the `pipelines` workbench, but as
ordinary notebooks — no `@dsl.component`, no `Compiler().compile(...)`, no
submit cell. Run them top to bottom, break them, change the parameters.

| Notebook | Reads | Writes |
|---|---|---|
| `01_clean_data` | the three raw downloads | `clean/branches.csv`, `clean/population_demographics.csv`, `clean/census_demos.csv` |
| `02_fetch_isochrones` | `clean/branches.csv` | `features/isochrones.parquet` |
| `03_compute_coverage` | isochrones + tracts | `results/coverage_details.parquet` |
| `04_aggregate_scores` | `results/coverage_details.parquet` | `results/coverage_scores.parquet` |
| `05_train_model` | details + scores + branches + tracts | `models/coverage-model/model.json`, `results/coverage_predictions.*` |

Every step hands off through S3, never through a variable in memory. That is
what makes them rearrangeable: a step does not care whether the one before it
ran five seconds ago in the next cell or last Tuesday in a pipeline.

## Building a pipeline out of them

**1. Make a canvas.** File → New → Pipeline Editor. Save it as something like
`coverage.pipeline` — Elyra writes a JSON file next to your notebooks.

**2. Drag the notebooks on** in order, and draw an arrow from each to the next.
The arrows are the *only* thing that sets execution order.

**3. Set node properties** — right-click a node → Open Properties. The two that
matter:

- **Runtime image** — pick `Datascience with Python 3.12`. Every notebook
  installs its own dependencies in the first cell, so the image mostly just
  needs pip.
- **Kubernetes secrets** — this is how the node gets its credentials. Every
  node needs four entries from `cpl-s3`:

  | Environment variable | Secret name | Secret key |
  |---|---|---|
  | `AWS_ACCESS_KEY_ID` | `cpl-s3` | `AWS_ACCESS_KEY_ID` |
  | `AWS_SECRET_ACCESS_KEY` | `cpl-s3` | `AWS_SECRET_ACCESS_KEY` |
  | `AWS_S3_ENDPOINT` | `cpl-s3` | `AWS_S3_ENDPOINT` |
  | `AWS_S3_BUCKET` | `cpl-s3` | `AWS_S3_BUCKET` |

  `02_fetch_isochrones` additionally needs `MAPBOX_TOKEN` from the
  `mapbox-token` secret.

**4. Run it.** The play button submits to the Data Science Pipelines server,
and the run shows up under Pipelines in the dashboard like any other.

## Three nodes, one notebook

`02_fetch_isochrones` reads `MODE` and `TRAVEL_TIME` from the environment
rather than hardcoding them. Drop the same notebook on the canvas three times
and give each copy different **Environment Variables** in its node properties:

| Node | `MODE` | `TRAVEL_TIME` |
|---|---|---|
| driving | `driving` | `5` |
| cycling | `cycling` | `10` |
| walking | `walking` | `10` |

Chain them in series rather than running them side by side — Mapbox allows 300
requests a minute and three parallel nodes will trip it.

`05_train_model` works the same way with `MODE`, `TRAVEL_TIME`,
`NEW_BRANCH_LAT` and `NEW_BRANCH_LON`, so you can fan out several candidate
locations and compare.

## Where this differs from the real pipelines

Two deliberate simplifications, both because an Elyra node cannot mount a
ConfigMap:

- **No Feast.** The KFP version checks a Feast online store before calling
  Mapbox. Here `02` does the same freshness check directly against the parquet
  cache in S3. The behaviour is the same; there is just no feature store
  recording it.
- **MLflow is optional.** This workbench has `MLFLOW_TRACKING_URI` injected, so
  running `05` interactively logs a run. An Elyra node does not get that
  injection, and `05` skips logging rather than failing.

Everything else — the same cleaning rules, the same intersection maths, the
same model and hyperparameters — is identical to what runs in production.

## If something breaks

- **`KeyError: 'AWS_S3_ENDPOINT'`** — the node is missing its secret mapping,
  or you are running interactively in a workbench that was started before the
  `cpl-s3` secret existed. Restart the workbench.
- **A later step says a key does not exist** — the step before it has not run
  yet. Check the bucket, not the notebook.
- **Mapbox 401** — `mapbox-token` is empty. It comes from `secrets.mapboxToken`
  in the chart's `values.yaml`.
- **Nothing shows up in the Pipelines UI after you hit run** — Elyra needs a
  runtime configuration. RHOAI creates one automatically for workbenches in a
  project that has a pipeline server; if the dropdown is empty, the pipeline
  server has not finished starting.